# Collect All Results into Zip File

This notebook gathers all experiment results, visualizations, checkpoints, and outputs into a single zip file for easy sharing or backup.

## 🚀 Google Colab Users

**If you're running this in Google Colab**, this notebook will:
- Automatically detect Colab environment
- Mount your Google Drive (if not already mounted)
- Look for results in your Drive (e.g., `/content/drive/MyDrive/NOT_results/`)
- Save the zip file directly to your Drive for easy download

**Setup for Colab:**
1. Make sure your experiment results are saved to Google Drive (e.g., `/content/drive/MyDrive/NOT_results/experiments/`)
2. Run all cells - the notebook will automatically handle Colab paths
3. The zip file will be saved to your Drive and you can download it from the Colab file browser

## What gets collected:
- JSON/CSV result files (results_summary.json, results_summary.csv, etc.)
- Visualization plots (PNG files)
- Model checkpoints (.ckpt files)
- TensorBoard logs (events.out.tfevents.*)
- Configuration files (hparams.yaml, configs)
- Intervention results
- Interpretation reports
- Any other output files from experiments


In [ ]:
import os
import zipfile
from pathlib import Path
from datetime import datetime
import json
from collections import defaultdict

# Detect if running in Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("🌐 Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("💻 Running locally")

# Colab-specific setup
if IN_COLAB:
    # Mount Google Drive if not already mounted
    drive_path = Path('/content/drive')
    if not drive_path.exists():
        print("📁 Mounting Google Drive...")
        drive.mount('/content/drive')
    else:
        print("✅ Google Drive already mounted")
    
    # Common Colab result locations
    possible_drive_paths = [
        Path('/content/drive/MyDrive/NOT_results'),
        Path('/content/drive/MyDrive/experiments'),
        Path('/content/drive/MyDrive/Negation-Origin-Tracing/experiments'),
    ]
    
    # Check if results are in Drive
    drive_experiments = None
    for path in possible_drive_paths:
        if path.exists():
            drive_experiments = path
            print(f"✅ Found results in Drive: {drive_experiments}")
            break
    
    if drive_experiments:
        EXPERIMENTS_DIR = drive_experiments
        OUTPUT_DIR = drive_experiments.parent / 'results_archive'
        PROJECT_ROOT = drive_experiments.parent
    else:
        # Fall back to local Colab directory
        print("⚠️  No results found in common Drive locations, using local directory")
        PROJECT_ROOT = Path('/content')
        if (PROJECT_ROOT / 'Negation-Origin-Tracing').exists():
            PROJECT_ROOT = PROJECT_ROOT / 'Negation-Origin-Tracing'
        EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
        OUTPUT_DIR = Path('/content/drive/MyDrive/results_archive')  # Save to Drive anyway
else:
    # Local setup
    PROJECT_ROOT = Path.cwd()
    if 'Negation-Origin-Tracing' in str(PROJECT_ROOT):
        # Already in project root
        pass
    elif (PROJECT_ROOT / 'Negation-Origin-Tracing').exists():
        PROJECT_ROOT = PROJECT_ROOT / 'Negation-Origin-Tracing'
        os.chdir(PROJECT_ROOT)
    
    EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
    OUTPUT_DIR = PROJECT_ROOT / 'results_archive'

print(f"\n📂 Configuration:")
print(f"   Project root: {PROJECT_ROOT}")
print(f"   Experiments directory: {EXPERIMENTS_DIR}")
print(f"   Output directory: {OUTPUT_DIR}")
print(f"   {'(Colab)' if IN_COLAB else '(Local)'}")

# Verify directories exist
if not EXPERIMENTS_DIR.exists():
    print(f"\n⚠️  WARNING: Experiments directory not found: {EXPERIMENTS_DIR}")
    print("   Make sure your results are saved in the correct location.")


In [ ]:
# File patterns to collect
RESULT_PATTERNS = {
    'json': ['**/*.json'],
    'csv': ['**/*.csv'],
    'plots': ['**/*.png', '**/*.jpg', '**/*.jpeg', '**/*.pdf', '**/*.svg'],
    'checkpoints': ['**/*.ckpt'],
    'logs': ['**/events.out.tfevents.*', '**/*.tfevents.*'],
    'configs': ['**/hparams.yaml', '**/*.yaml', '**/*.yml'],
    'reports': ['**/*.md', '**/*.txt'],
}

# Directories to include
# For Colab, we search in the experiments directory directly
# For local, we search in project subdirectories
if IN_COLAB:
    # In Colab, EXPERIMENTS_DIR might be the full path to results
    INCLUDE_DIRS = [str(EXPERIMENTS_DIR)]
else:
    INCLUDE_DIRS = [
        'experiments',
        'notebooks',  # Include notebooks themselves
    ]

# Directories/files to exclude
EXCLUDE_PATTERNS = [
    '__pycache__',
    '*.pyc',
    '.git',
    'node_modules',
    '.ipynb_checkpoints',
    '*.ipynb',  # Exclude notebooks from zip (they're large, include separately if needed)
]

# Exclude checkpoints to save space (they can be regenerated)
# Set to False if you want to include checkpoints
EXCLUDE_CHECKPOINTS = True  # Default: exclude checkpoints

if EXCLUDE_CHECKPOINTS:
    print("✅ Checkpoints will be excluded from archive (to save space)")
else:
    print("⚠️  Checkpoints will be included in archive (may create large zip file)")

print("Configuration loaded.")


In [ ]:
def should_exclude(file_path: Path) -> bool:
    """Check if a file should be excluded."""
    path_str = str(file_path)
    
    for pattern in EXCLUDE_PATTERNS:
        if pattern in path_str:
            return True
    
    return False


def find_files(base_dir: Path, patterns: list) -> list:
    """Find all files matching patterns in base_dir."""
    files = []
    
    for pattern in patterns:
        # Handle glob patterns
        if '**' in pattern:
            # Recursive search
            found = list(base_dir.glob(pattern))
        else:
            # Non-recursive
            found = list(base_dir.glob(pattern))
        
        for f in found:
            if f.is_file() and not should_exclude(f):
                files.append(f)
    
    return files


def collect_all_results() -> dict:
    """Collect all result files organized by type."""
    all_files = defaultdict(list)
    
    print("Scanning for result files...")
    
    for dir_path_str in INCLUDE_DIRS:
        if IN_COLAB:
            # In Colab, INCLUDE_DIRS contains full paths
            dir_path = Path(dir_path_str)
        else:
            dir_path = PROJECT_ROOT / dir_path_str
        
        if not dir_path.exists():
            print(f"  ⚠️  Directory not found: {dir_path}")
            continue
        
        print(f"\n  Scanning {dir_path.name if IN_COLAB else dir_path_str}/...")
        
        for file_type, patterns in RESULT_PATTERNS.items():
            # Skip checkpoints if EXCLUDE_CHECKPOINTS is True
            if file_type == 'checkpoints' and EXCLUDE_CHECKPOINTS:
                continue
                
            files = find_files(dir_path, patterns)
            all_files[file_type].extend(files)
            if files:
                print(f"    {file_type}: {len(files)} files")
    
    # Also check for any other important files in experiments directory
    if EXPERIMENTS_DIR.exists():
        print(f"\n  Scanning {EXPERIMENTS_DIR.name}/ for any additional files...")
        
        # Find all files in experiments directory
        for root, dirs, files in os.walk(EXPERIMENTS_DIR):
            # Skip excluded directories
            dirs[:] = [d for d in dirs if not should_exclude(Path(root) / d)]
            
            for file in files:
                file_path = Path(root) / file
                
                if should_exclude(file_path):
                    continue
                
                # Skip checkpoints if excluded
                if file_path.suffix == '.ckpt' and EXCLUDE_CHECKPOINTS:
                    continue
                
                # Check if already categorized
                already_categorized = any(file_path in all_files[ft] for ft in all_files)
                
                if not already_categorized:
                    # Categorize by extension
                    ext = file_path.suffix.lower()
                    if ext in ['.json', '.csv', '.yaml', '.yml', '.ckpt', '.png', '.jpg', '.pdf', '.svg']:
                        # Already handled by patterns
                        pass
                    elif ext in ['.txt', '.md', '.log']:
                        all_files['reports'].append(file_path)
                    elif 'tfevents' in file_path.name:
                        all_files['logs'].append(file_path)
    
    return all_files


# Collect all files
collected_files = collect_all_results()

# Print summary
print("\n" + "="*60)
print("COLLECTION SUMMARY")
print("="*60)
total_files = 0
for file_type, files in collected_files.items():
    count = len(files)
    total_files += count
    print(f"{file_type:15s}: {count:4d} files")
print("-"*60)
print(f"{'TOTAL':15s}: {total_files:4d} files")


In [ ]:
def create_zip_archive(files_dict: dict, output_path: Path) -> None:
    """Create a zip archive with all collected files."""
    
    # Create output directory
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    print(f"\nCreating zip archive: {output_path}")
    print("This may take a while for large files...")
    
    total_size = 0
    files_added = 0
    
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        
        # Add files organized by type
        for file_type, files in files_dict.items():
            if not files:
                continue
            
            print(f"\n  Adding {file_type} files...")
            
            for file_path in files:
                try:
                    # Calculate relative path from project root
                    try:
                        rel_path = file_path.relative_to(PROJECT_ROOT)
                    except ValueError:
                        # File is outside project root, use full path structure
                        rel_path = Path('external') / file_path.name
                    
                    # Organize in zip by file type
                    zip_path = f"{file_type}/{rel_path}"
                    
                    # Add file to zip
                    zipf.write(file_path, zip_path)
                    
                    file_size = file_path.stat().st_size
                    total_size += file_size
                    files_added += 1
                    
                    if files_added % 50 == 0:
                        print(f"    Added {files_added} files... ({total_size / 1024 / 1024:.1f} MB)")
                        
                except Exception as e:
                    print(f"    ⚠️  Error adding {file_path}: {e}")
        
        # Add a manifest file
        manifest = {
            'created_at': datetime.now().isoformat(),
            'project_root': str(PROJECT_ROOT),
            'file_counts': {ft: len(files) for ft, files in files_dict.items()},
            'total_files': files_added,
            'total_size_bytes': total_size,
            'total_size_mb': total_size / 1024 / 1024,
        }
        
        manifest_str = json.dumps(manifest, indent=2)
        zipf.writestr('MANIFEST.json', manifest_str)
    
    # Print final summary
    zip_size = output_path.stat().st_size
    print(f"\n" + "="*60)
    print("ZIP ARCHIVE CREATED")
    print("="*60)
    print(f"Output: {output_path}")
    print(f"Files added: {files_added}")
    print(f"Total size (uncompressed): {total_size / 1024 / 1024:.2f} MB")
    print(f"Zip file size: {zip_size / 1024 / 1024:.2f} MB")
    print(f"Compression ratio: {total_size / zip_size:.2f}x")
    print(f"\nManifest saved in: MANIFEST.json")


In [ ]:
# Create timestamped zip file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"negation_origin_tracing_results_{timestamp}.zip"
zip_path = OUTPUT_DIR / zip_filename

# Create the archive
create_zip_archive(collected_files, zip_path)

print(f"\n✅ Archive created successfully!")
print(f"   Location: {zip_path}")

# Colab-specific: Show download instructions
if IN_COLAB:
    print(f"\n📥 To download in Colab:")
    print(f"   1. Open the file browser (folder icon on left sidebar)")
    print(f"   2. Navigate to: {OUTPUT_DIR}")
    print(f"   3. Right-click '{zip_filename}' and select 'Download'")
    print(f"\n   Or use this command to download directly:")
    print(f"   from google.colab import files")
    print(f"   files.download('{zip_path}')")


In [ ]:
# Optional: Create a detailed file listing
listing_path = OUTPUT_DIR / f"file_listing_{timestamp}.txt"

with open(listing_path, 'w') as f:
    f.write("NEGATION ORIGIN TRACING - RESULTS ARCHIVE\n")
    f.write("="*60 + "\n\n")
    f.write(f"Created: {datetime.now().isoformat()}\n")
    f.write(f"Project Root: {PROJECT_ROOT}\n\n")
    
    f.write("FILE LISTING BY TYPE\n")
    f.write("-"*60 + "\n\n")
    
    for file_type, files in collected_files.items():
        if not files:
            continue
        
        f.write(f"\n{file_type.upper()} ({len(files)} files):\n")
        f.write("-"*60 + "\n")
        
        for file_path in sorted(files):
            try:
                rel_path = file_path.relative_to(PROJECT_ROOT)
                size = file_path.stat().st_size
                f.write(f"  {rel_path} ({size / 1024:.1f} KB)\n")
            except:
                f.write(f"  {file_path}\n")

print(f"\n📄 Detailed file listing saved to: {listing_path}")


In [ ]:
# Optional: Show some statistics about what was collected
print("\n" + "="*60)
print("COLLECTION STATISTICS")
print("="*60)

for file_type, files in collected_files.items():
    if not files:
        continue
    
    total_size = sum(f.stat().st_size for f in files)
    avg_size = total_size / len(files) if files else 0
    
    print(f"\n{file_type.upper()}:")
    print(f"  Files: {len(files)}")
    print(f"  Total size: {total_size / 1024 / 1024:.2f} MB")
    print(f"  Average size: {avg_size / 1024:.1f} KB")
    
    # Show top 5 largest files
    sorted_files = sorted(files, key=lambda f: f.stat().st_size, reverse=True)[:5]
    if sorted_files:
        print(f"  Largest files:")
        for f in sorted_files:
            try:
                rel_path = f.relative_to(PROJECT_ROOT)
                size_mb = f.stat().st_size / 1024 / 1024
                print(f"    {rel_path}: {size_mb:.2f} MB")
            except:
                print(f"    {f.name}: {f.stat().st_size / 1024 / 1024:.2f} MB")


## Next Steps

The zip file has been created with all your results. You can:

1. **Share the zip file** with collaborators or for backup
2. **Extract and explore** the results in the organized folder structure
3. **Check the MANIFEST.json** file inside the zip for metadata
4. **Review the file listing** text file for a detailed inventory

### 📥 Downloading (Google Colab)

If you're in Colab, the zip file is saved to your Google Drive. To download:

**Option 1: File Browser**
1. Click the folder icon (📁) in the left sidebar
2. Navigate to the `results_archive` folder in your Drive
3. Right-click the zip file and select "Download"

**Option 2: Direct Download (run cell below)**
```python
from google.colab import files
files.download(str(zip_path))
```

### Zip File Structure

The zip file is organized by file type:
- `json/` - All JSON result files
- `csv/` - CSV result summaries
- `plots/` - Visualization images (PNG, PDF, SVG)
- `checkpoints/` - Model checkpoints (.ckpt files)
- `logs/` - TensorBoard log files
- `configs/` - Configuration files (YAML)
- `reports/` - Text reports and markdown files

### Note on Large Files

If the zip file is very large (>100MB), you may want to:
- Set `EXCLUDE_CHECKPOINTS = True` in the configuration cell (excludes .ckpt files)
- Exclude TensorBoard logs (they can be regenerated)
- Only include final results, not intermediate checkpoints

To customize what gets included, modify the `RESULT_PATTERNS` and `EXCLUDE_PATTERNS` in the configuration cell above.


In [ ]:
# Optional: Direct download for Colab users
if IN_COLAB:
    print("📥 Downloading zip file directly...")
    try:
        from google.colab import files
        files.download(str(zip_path))
        print("✅ Download started!")
    except Exception as e:
        print(f"⚠️  Download failed: {e}")
        print(f"   You can still download from the file browser: {zip_path}")
else:
    print("💻 Local mode: Zip file is saved to your local filesystem")
    print(f"   Location: {zip_path}")
